# 2. Dedisperse the acquisition and read the maps

**Scenario:** the pulse is spread across time in the spectrogram. You now want PIRATE to compensate for dispersion across a range of trial dispersion measures (DMs), then inspect where the signal becomes concentrated.

Use the saved frames from [notebook 1](01_injection.ipynb). Start the Bash terminal and Jupyter kernel in this tutorial repository. Set the blank RUN_DIR and run_directory fields to the **same full run-folder path used in notebook 1**.

Read the config, run the command in your activated GPU terminal, and wait for it to finish before running the Python cells. These cells only read and plot its outputs. The instructor has reported a successful local walkthrough; this standalone edition has not been rerun.

## 1. Read the saved dedispersion configuration

Open dedispersion.yml inside your run folder, copied from [configs/dedispersion.yml](configs/dedispersion.yml). It retains the acquisition's native frequency channels and 0.9984 ms sampling. The important search settings are:

~~~yaml
dtype: float16
time_samples_per_chunk: 2048
frequency_subband_counts: [0, 0, 0, 0, 1]
toplevel_tree_rank: 16
primary_trees:
  - {num_early_triggers: 0, max_width: 32, wt_dm_downsampling: 64, wt_time_downsampling: 64}
~~~

For this tutorial, these select **one full-band search tree**, approximately covering DM 0-1479 pc cm^-3. A tree is PIRATE's organized set of dedispersion trials. Leave these structural settings unchanged for now.

PIRATE also applies temporal filters of different widths and shapes to combine nearby time samples. It searches boxcar, triangular, and trapezoidal profiles, so its recovered nominal filter width is not necessarily a boxcar width or the injected Gaussian sigma.

## 2. Run dedispersion in the terminal

Fill in RUN_DIR with the same full run-folder path used in notebook 1, then run this command in your activated GPU terminal. The ${RUN_DIR:?...} check prevents running with a blank path.

~~~bash
RUN_DIR=""  # REQUIRED: full path to the existing run folder from notebook 1.
python -m pirate_frb run offline_dedisperser \
    "${RUN_DIR:?Fill in RUN_DIR first}/frames" "$RUN_DIR/dedispersion.yml" --save
~~~

Process **all four chunks in order**. The dedisperser uses earlier input data to form later outputs. Wait until the command finishes without error before continuing.

The command prints each chunk's maximum S/N and saves frame_b1_tN_snrmap.asdf **beside each intensity frame, inside frames/**. Keep the maps there for notebook 3. Rerunning the command replaces those map files. Reuse the same input frames when comparing processing so their noise realization stays unchanged.

This stage does not write a grouped candidate catalog. Notebook 3 performs that step.


## 3. Understand out_max and out_argmax

The GPU dedisperser produces two matching arrays for each tree:

| Native output | Question it answers | Saved ASDF field |
| --- | --- | --- |
| out_max | How strong was the best trial in this coarse DM-time pixel? | trees[0]['snr'] |
| out_argmax | Which fine trial and filter produced that maximum? | trees[0]['argmax'] |

**out_max is a maximum, not an average.** Each coarse pixel retains the largest normalized S/N over the finer time/DM positions, temporal profiles, and configured frequency-subband trials belonging to it. Here the only enabled frequency subband is the full band.

**out_argmax is an encoded integer token**, stored as uint32. Its matching pixel identifies the winning fine trial. The integer's size is not a brightness, DM, time, or width: a token heatmap would not show a physical quantity.

The native arrays have a beam axis. For this one-beam run their expected shape is (1, 4096, 128); the saved file removes the beam axis, leaving two arrays of shape (4096, 128), ordered **DM rows, then time columns**.

To recover physical values, PIRATE needs the token, pixel indices, source chunk, and the exact saved producer metadata. The grouper handles this decoding in notebook 3. These maxima and tokens do not retain every discarded fine trial or a dedispersed frequency-time waterfall.

A within-pixel maximum is **not yet a detection**. The search will compare neighboring pixels and apply its selection rules.

## 4. Read the map containing the pulse

The default command in notebook 1 injected DM 20 pc cm^-3 with a 5 s arrival at 300 MHz; its pulse lies in chunk 2. Consult that run's generation.log if you changed the experiment.

A displayed column spans about 16 ms and retains a maximum from finer trials. We therefore label the axis **coarse output time**, not exact pulse-center arrival time. The coordinate arithmetic below only labels the plot.

In [ ]:
from pathlib import Path
import asdf
import matplotlib.pyplot as plt
import numpy as np


In [ ]:
run_directory = ""  # REQUIRED: full path to the run folder you chose in the terminal.
if not run_directory.strip():
    raise ValueError("Fill in run_directory above before continuing.")
run_dir = Path(run_directory).expanduser()
map_path = run_dir / 'frames/frame_b1_t2_snrmap.asdf'


In [ ]:
with asdf.open(map_path) as saved:
    tree = saved.tree['trees'][0]
    snr = np.asarray(tree['snr']).copy()
    argmax = np.asarray(tree['argmax']).copy()
    dm_edges = np.linspace(tree['dm_min'], tree['dm_max'], snr.shape[0] + 1)
    dt = saved.tree['time_sample_ms'] * 1e-3
    nt_in = saved.tree['nt_in']
    chunk = saved.tree['source']['time_chunk_index']
time_edges = (chunk * nt_in + np.arange(snr.shape[1] + 1) * nt_in / snr.shape[1]) * dt
print(f'S/N map: {snr.shape}, {snr.dtype}; matching tokens: {argmax.shape}, {argmax.dtype}')

In [ ]:
injected_dm = 20.0  # Plot annotation from the baseline command; see generation.log.
fig, ax = plt.subplots(figsize=(9, 4), constrained_layout=True)
image = ax.pcolormesh(time_edges, dm_edges, snr, shading='flat', cmap='magma', rasterized=True)
ax.axhline(injected_dm, color='cyan', linestyle='--', label='Injected DM')
ax.set(xlabel='Coarse output time since observation start (s)',
       ylabel='Trial DM (pc cm$^{-3}$)', ylim=(0, 60),
       title='PIRATE S/N map: a zoom in DM')
ax.legend()
fig.colorbar(image, ax=ax, label='Filtered S/N')
plt.show()

**What to look for:** a bright response near the injected DM and arrival time, surrounded by a broader pattern. The plot only shows DM 0-60; the configured search covered the larger range.

This short observation can lack enough earlier data for high-DM trials. Our low-DM signal is placed well inside the observation; quality flags in the eventual catalog still need inspection.

## 5. Compare a matching and an incorrect DM

Take two horizontal slices of the same saved S/N map. A matching DM should produce a sharper, stronger response. At an incorrect DM, channels fail to line up, so the response can weaken, broaden, and shift in time.

These are **filtered S/N profiles**, not raw intensity profiles. Each label gives the DM interval represented by the selected row. The coarse map columns do not directly resolve the injected 3 ms Gaussian sigma.

In [ ]:
wrong_dm = 40.0
time_centers = (time_edges[:-1] + time_edges[1:]) / 2
fig, ax = plt.subplots(figsize=(9, 3), constrained_layout=True)
for trial_dm, label in [(injected_dm, 'Matching'), (wrong_dm, 'Incorrect')]:
    row = np.searchsorted(dm_edges, trial_dm, side='right') - 1
    ax.plot(time_centers, snr[row],
            label=f'{label}: DM [{dm_edges[row]:.2f}, {dm_edges[row + 1]:.2f}) pc cm^-3')
ax.set(xlabel='Coarse output time since observation start (s)',
       ylabel='Filtered S/N', title='Two slices through the same PIRATE map')
ax.legend()
plt.show()

### Try one change

Change wrong_dm from 40 to 25 in the final plotting cell, then rerun only that cell. Predict whether the response will resemble the matching-DM curve more closely. Keep this display value between 0 and 60 for the exercise.

**No terminal command needs rerunning:** these trials are already in the saved map. Changing a plotted row does not change the search configuration or regenerate data.

Continue with [notebook 3](03_search.ipynb), where a terminal command turns these maps and their tokens into detections and grouped candidates.